In [10]:
from pathlib import Path
import pandas as pd

In [11]:
project_root = Path("/Users/lipikam/Desktop/MinorProject/ANSC4040MiniProject")
data_file_path = project_root / "Data_set_prep_assignment_1.csv"
output_file_path = project_root / "MiniProjectMetaData.xlsx"

df = pd.read_csv(data_file_path, low_memory=False, dtype={"AnimalId": "string"})
df = df.rename(columns={
    "Avgmilkflow": "AverageMilkFlowKgPerMin",
    "Flow30_60Session": "MilkFlow30To60SecondsKgPerMin",
    "YieldFirst2Min_Session": "MilkYieldFirst2MinutesKg",
    "YieldSession": "TotalMilkYieldSessionKg",
    "DurationSession_sec": "MilkingDurationSessionSeconds",
    "milking": "MilkingSession"
})
df["EventDate"] = pd.to_datetime(df["EventDate"], errors="coerce")

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
display(df.head())

Rows: 8,495,421
Columns: 11


,AnimalId,LactationNumber,DaysInMilk,ReproductionStatus,EventDate,AverageMilkFlowKgPerMin,MilkFlow30To60SecondsKgPerMin,MilkYieldFirst2MinutesKg,TotalMilkYieldSessionKg,MilkingDurationSessionSeconds,MilkingSession
0,-8839528597343470980,1.0,365.0,Pregnant,2019-12-09,2.902991,0.898113,4.975908,11.158372,226,1
1,-5365332022005618386,1.0,66.0,Bred,2021-07-12,3.311224,1.401600,5.347854,15.059267,268,2
2,<NA>,NaN,NaN,NaN,2020-08-24,3.220506,3.501733,7.547777,14.560315,270,1
3,7750423760892252046,1.0,244.0,Pregnant,2019-12-04,3.900894,4.100475,8.400531,15.331422,231,3
4,<NA>,NaN,NaN,NaN,2021-03-21,4.218409,5.098378,9.198853,13.970645,198,2


In [12]:
definitions = {
    "AnimalId": "Animal identifier",
    "LactationNumber": "Lactation number",
    "DaysInMilk": "Days since calving",
    "ReproductionStatus": "Reproductive-status category",
    "EventDate": "Date of the milking event",
    "AverageMilkFlowKgPerMin": "Average milk flow rate",
    "MilkFlow30To60SecondsKgPerMin": "Flow rate from 30-60 seconds into milking",
    "MilkYieldFirst2MinutesKg": "Milk produced during the first 2 minutes",
    "TotalMilkYieldSessionKg": "Total milk produced during the session",
    "MilkingDurationSessionSeconds": "Session duration",
    "MilkingSession": "Milking/session indicator or category"
}
canonical_units = {
    "AnimalId": "Identifier", "LactationNumber": "Count", "DaysInMilk": "Days",
    "ReproductionStatus": "Category", "EventDate": "Date",
    "AverageMilkFlowKgPerMin": "kg/min", "MilkFlow30To60SecondsKgPerMin": "kg/min",
    "MilkYieldFirst2MinutesKg": "kg", "TotalMilkYieldSessionKg": "kg",
    "MilkingDurationSessionSeconds": "seconds", "MilkingSession": "Category"
}

metadata_rows = []
for column_name in df.columns:
    series = df[column_name]
    if pd.api.types.is_numeric_dtype(series):
        minimum, maximum = series.min(), series.max()
        value_range = maximum - minimum
    elif pd.api.types.is_datetime64_any_dtype(series):
        minimum = series.min().date().isoformat()
        maximum = series.max().date().isoformat()
        value_range = (series.max() - series.min()).days
    else:
        minimum = maximum = value_range = "N/A"

    metadata_rows.append({
        "Current name": column_name,
        "Suggested name": column_name,
        "Definition": definitions[column_name],
        "Total count": len(series),
        "Non-missing count": series.notna().sum(),
        "Missing count": series.isna().sum(),
        "Missing percent": round(series.isna().mean() * 100, 2),
        "Data type": str(series.dtype),
        "Unique values": series.nunique(dropna=True),
        "Minimum": minimum,
        "Maximum": maximum,
        "Range": value_range,
        "Canonical unit": canonical_units[column_name],
        "Example values": ", ".join(series.dropna().astype(str).unique()[:5])
    })

df_metadata_completed = pd.DataFrame(metadata_rows)
display(df_metadata_completed)

,Current name,Suggested name,Definition,Total count,Non-missing count,Missing count,Missing percent,Data type,Unique values,Minimum,Maximum,Range,Canonical unit,Example values
0,AnimalId,AnimalId,Animal identifier,8495421,6794418,1701003,20.02,string,9087,N/A,N/A,N/A,Identifier,"-8839528597343470980, -5365332022005618386, 77..."
1,LactationNumber,LactationNumber,Lactation number,8495421,6794418,1701003,20.02,float64,12,1.0,12.0,11.0,Count,"1.0, 5.0, 4.0, 2.0, 3.0"
2,DaysInMilk,DaysInMilk,Days since calving,8495421,6794414,1701007,20.02,float64,870,1.0,870.0,869.0,Days,"365.0, 66.0, 244.0, 394.0, 288.0"
3,ReproductionStatus,ReproductionStatus,Reproductive-status category,8495421,6794418,1701003,20.02,str,4,N/A,N/A,N/A,Category,"Pregnant, Bred, Open, Fresh"
4,EventDate,EventDate,Date of the milking event,8495421,8495421,0,0.00,datetime64[us],830,2019-06-14,2021-10-24,863,Date,"2019-12-09, 2021-07-12, 2020-08-24, 2019-12-04..."
5,AverageMilkFlowKgPerMin,AverageMilkFlowKgPerMin,Average milk flow rate,8495421,8495249,172,0.00,float64,219,0.0,23.496085,23.496085,kg/min,"2.902991168, 3.311224301, 3.220505827, 3.90089..."
6,MilkFlow30To60SecondsKgPerMin,MilkFlow30To60SecondsKgPerMin,Flow rate from 30-60 seconds into milking,8495421,8495421,0,0.00,float64,102,0.0,10.500663,10.500663,kg/min,"0.8981128926, 1.4016004233, 3.5017330964, 4.10..."
7,MilkYieldFirst2MinutesKg,MilkYieldFirst2MinutesKg,Milk produced during the first 2 minutes,8495421,8495421,0,0.00,float64,1675,2.000342,11.997518,9.997176,kg,"4.9759082989, 5.3478540423, 7.5477770368, 8.40..."
8,TotalMilkYieldSessionKg,TotalMilkYieldSessionKg,Total milk produced during the session,8495421,8495421,0,0.00,float64,1010,5.034875,54.839318,49.804442,kg,"11.158372302000002, 15.059266684000002, 14.560..."
9,MilkingDurationSessionSeconds,MilkingDurationSessionSeconds,Session duration,8495421,8495421,0,0.00,int64,604,100,785,685,seconds,"226, 268, 270, 231, 198"


In [ ]:
df_metadata_completed.to_excel(output_file_path, sheet_name="Metadata", index=False)
print(f"Saved completed metadata to:\n{output_file_path}")